# CoordinX - Organic Account Screening

Notebook ini menyaring **weak Organic labels** dari seluruh data final secara konservatif.

Kebijakan: label manual tetap menjadi gold label; akun yang tidak memenuhi semua sinyal tidak dipaksa menjadi Organic dan tetap `UNKNOWN`. Batch 4-5 dipindai tetapi tidak diberi weak Organic otomatis karena kemiripan template lintas akun sangat dominan. Batch 6 tidak dipakai karena overlap dengan Batch 4.

In [1]:
from pathlib import Path
import json
import os
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "PRD.md").exists())
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")
print("Repository:", ROOT)
print("Notebook setup: OK")

Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-
Notebook setup: OK


In [2]:
from preprocessing.organic_screening import run_screening
summary = run_screening(ROOT)
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "schema_version": "organic-screening-v1",
  "generated_at_utc": "2026-11-11T13:29:37.065591+00:00",
  "batches_scanned": [
    "batch2",
    "batch3",
    "batch4",
    "batch5"
  ],
  "eligible_batches": [
    "batch2",
    "batch3"
  ],
  "excluded_from_automatic_organic": {
    "batch4": "Template similarity is pervasive; no automatic Organic weak label added.",
    "batch5": "Template similarity is pervasive; no automatic Organic weak label added.",
    "batch6": "Near-duplicate of batch4 and excluded from independent final training."
  },
  "accounts_scanned": 3689,
  "manual_labeled_accounts": 200,
  "organic_candidates": 217,
  "candidates_by_batch": {
    "batch2": 135,
    "batch3": 82,
    "batch4": 0,
    "batch5": 0
  },
  "candidate_policy": {
    "min_posts": 2,
    "max_peer_similarity": 0.6,
    "max_burst_ratio_1h": 0.5,
    "min_active_span_hours_or_days": "1 hour or 2 calendar days",
    "min_unique_text_ratio": 0.75,
    "sample_weight": 0.35
  },
  "artifacts":

In [3]:
import pandas as pd

organic_path = ROOT / "data" / "labels" / "final_organic_screened.csv"
audit_path = ROOT / "data" / "labels" / "final_organic_screening_audit.csv"
organic = pd.read_csv(organic_path, dtype="string")
audit = pd.read_csv(audit_path, dtype="string")
print("Organic weak labels:", len(organic))
print(organic.groupby("dataset_id").size().to_dict())
print("Audit rows:", len(audit))
organic[["username", "dataset_id", "n_posts", "max_peer_similarity", "burst_ratio_1h", "evidence"]].head(10)

Organic weak labels: 217
{'batch2': 135, 'batch3': 82}
Audit rows: 3689


,username,dataset_id,n_posts,max_peer_similarity,burst_ratio_1h,evidence
0,Uniliswara0,batch2,3,0.27679711864756,0.0,Supplemental Organic weak label: 3 post dalam ...
1,Ar_910,batch2,3,0.18856127049664279,0.0,Supplemental Organic weak label: 3 post dalam ...
2,BuchroniK,batch2,2,0.3710273360698382,0.0,Supplemental Organic weak label: 2 post dalam ...
3,mediaindonesia,batch2,4,0.44891186975216585,0.0,Supplemental Organic weak label: 4 post dalam ...
4,ceritaolyn8,batch2,7,0.4780607419597721,0.3333333333333333,Supplemental Organic weak label: 7 post dalam ...
5,seafishxx,batch2,5,0.24690428705513398,0.0,Supplemental Organic weak label: 5 post dalam ...
6,NHenryp29864,batch2,3,0.2566358510055926,0.0,Supplemental Organic weak label: 3 post dalam ...
7,kiissedbypoem,batch2,3,0.2590691958459504,0.0,Supplemental Organic weak label: 3 post dalam ...
8,PakarINTELek,batch2,2,0.26353122158004494,0.0,Supplemental Organic weak label: 2 post dalam ...
9,democrazymedia,batch2,6,0.36600297857492203,0.0,Supplemental Organic weak label: 6 post dalam ...


In [4]:
assert summary["accounts_scanned"] == len(audit)
assert summary["organic_candidates"] == len(organic)
assert summary["candidates_by_batch"]["batch4"] == 0
assert summary["candidates_by_batch"]["batch5"] == 0
assert organic["label"].astype(int).eq(0).all()
assert organic["review_status"].eq("AUTO_SCREENED_WEAK").all()
assert organic[["dataset_id", "account_id"]].duplicated().sum() == 0
print("Organic screening validation: OK")

Organic screening validation: OK
